In [1]:
from st.data import PriceData, ReturnData, CorrelationData
from st.dto.portfolio import *
from st.dto.position import *
from st.dto.strategy import *

## Collect Data and do Analytics
---

In [2]:
# ETF tickers - selected for low correlation
TICKERS = {
    'SPY': 'US Equities',
    'TLT': 'Long-Term Treasuries',
    'GLD': 'Gold',
    'DBC': 'Commodities',
    'VNQ': 'Real Estate'
}

price_datas = {tkr: PriceData(ticker=tkr) for tkr in TICKERS.keys()}
returns_datas = {tkr: ReturnData(price_data=pdata) for tkr, pdata in price_datas.items()}
correlation = CorrelationData(price_datas=list(price_datas.values()))

In [3]:
correlation.plot(height=300, width=400)

## Setup strategies
---

In [4]:
forecast_config = ForecastConfig(
    target_abs_forecast=10.0,
    min_forecast=-20.0,
    max_forecast=20.0,
    cap_forecasts=True,
    use_volatility_standardization=True
)
# Store strategies per instrument
instrument_strategies = {}

for ticker, price_data in price_datas.items():
    strategies = {}
    # EWMAC 16/64 (fast trend following)
    ewmac_fast = EWMACStrategyDTO(
        price_data=price_data,
        fast_span=16,
        slow_span=64,
        forecast_config=forecast_config
    )
    strategies['EWMAC_16_64'] = ewmac_fast

    # eqma slow
    ewmac_slow = EWMACStrategyDTO(
        price_data=price_data,
        fast_span=32,
        slow_span=128,
        forecast_config=forecast_config
    )
    strategies['EWMAC_32_128'] = ewmac_slow

    # Carry strategy (for non-equity assets)
    carry = CarryStrategyDTO(
        price_data=price_data,
        smoothing_span=30,
        forecast_config=forecast_config
    )
    strategies['Carry_30'] = carry

    # Mean reversion (cautious - can conflict with trend following)
    # mean_rev = MeanReversionStrategyDTO(
    #     price_data=price_data,
    #     lookback=20,
    #     entry_threshold=2.0,
    #     forecast_config=forecast_config
    # )
    # strategies['MeanRev_20'] = mean_rev

    # Turtle breakout
    turtle = TurtleStrategyDTO(
        price_data=price_data,
        entry_window=20,
        exit_window=10,
        forecast_config=forecast_config
    )
    strategies['Turtle_20_10'] = turtle
    instrument_strategies[ticker] = strategies


### Combine Forecasts

In [5]:
combined_forecasts = {}

for ticker, strategies in instrument_strategies.items():

    # Combine forecasts with auto-filtering of highly correlated strategies
    combined = CombinedForecastDTO(
        strategies=strategies,
        forecast_config=forecast_config,
        auto_filter_correlated=False  # Remove redundant strategies
    )

    combined_forecasts[ticker] = combined

    # Show forecast correlation matrix
    if len(combined.strategies) > 1:
        print(f"  Forecast correlations ({ticker}):")
        print(combined.forecast_correlation.round(3).to_string().replace('\n', '\n    '))


  Forecast correlations (SPY):
              EWMAC_16_64  EWMAC_32_128  Carry_30  Turtle_20_10
    EWMAC_16_64         1.000         0.871     0.903         0.515
    EWMAC_32_128        0.871         1.000     0.778         0.364
    Carry_30            0.903         0.778     1.000         0.324
    Turtle_20_10        0.515         0.364     0.324         1.000
  Forecast correlations (TLT):
              EWMAC_16_64  EWMAC_32_128  Carry_30  Turtle_20_10
    EWMAC_16_64         1.000         0.818     0.905         0.472
    EWMAC_32_128        0.818         1.000     0.754         0.246
    Carry_30            0.905         0.754     1.000         0.307
    Turtle_20_10        0.472         0.246     0.307         1.000
  Forecast correlations (GLD):
              EWMAC_16_64  EWMAC_32_128  Carry_30  Turtle_20_10
    EWMAC_16_64         1.000         0.835     0.890         0.494
    EWMAC_32_128        0.835         1.000     0.753         0.282
    Carry_30            0.890      

## Build Portfolio
---

In [6]:
INITIAL_CAPITAL = 10_000  # $100k
TARGET_VOLATILITY = 0.20  # 20% annual volatility target
BUFFER_FRACTION = 0.10  # 10% position change threshold

portfolio_risk = PortfolioRiskTargetDTO(
    annual_volatility_target=TARGET_VOLATILITY,
    notional_trading_capital=INITIAL_CAPITAL
)

# Create instruments
instruments = {}
for ticker in combined_forecasts.keys():
    instrument = InstrumentDTO(
        ticker=ticker,
        price_data=price_datas[ticker],
        combined_forecast=combined_forecasts[ticker],
        fx_rate=1.0  # All USD-denominated
    )
    instruments[ticker] = instrument

# Build portfolio with equal weighting
portfolio = PortfolioDTO(
    instruments=instruments,
    portfolio_risk_target=portfolio_risk,
    weighting_method='equal'
)

# Print portfolio summary
print(portfolio.get_portfolio_summary())


PORTFOLIO SUMMARY

Instruments: 5
Weighting: equal
IDM: 2.2584
Diversification: 100.0%

Risk Budget:
  Target vol:   20.00%
  Expected vol: 5.43%
  Ratio:        0.27x

Instrument Weights:
  SPY     :  20.0%  (avg |pos| =  16.19)
  TLT     :  20.0%  (avg |pos| =  28.23)
  GLD     :  20.0%  (avg |pos| =  17.83)
  DBC     :  20.0%  (avg |pos| = 109.80)
  VNQ     :  20.0%  (avg |pos| =  48.17)

Correlation Matrix:
        SPY   TLT   GLD   DBC   VNQ
  SPY  1.00 -0.03  0.00  0.02  0.01
  TLT -0.03  1.00 -0.03  0.00 -0.00
  GLD  0.00 -0.03  1.00  0.00 -0.02
  DBC  0.02  0.00  0.00  1.00 -0.01
  VNQ  0.01 -0.00 -0.02 -0.01  1.00



## Calculate Positions with Buffering
---

In [8]:
position_histories = {}
final_positions = {}
df = {}

for ticker, instrument in portfolio.instruments.items():
    # Get the basic position from portfolio (no buffering yet)
    portfolio_position = portfolio.get_position(ticker)

    # Apply buffering and rounding pipeline
    position_pipeline = PositionPipelineDTO(
        combined_forecast=instrument.combined_forecast.combined_forecast,
        instrument_volatility=instrument.volatility.annual_vol,
        price=instrument.price_data.data['Close'],
        portfolio_risk_target=portfolio_risk,
        instrument_weight=portfolio.instrument_weights.weights[ticker],
        idm=portfolio.idm_calculator.idm,
        fx_rate=instrument.fx_rate,
        current_position=None,  # Starting fresh (no existing positions)
        buffer_fraction=BUFFER_FRACTION,
        min_position_size=1.0
    )

    position_histories[ticker] = position_pipeline.final_position
    final_positions[ticker] = position_pipeline.final_position.iloc[-1]

    # Statistics
    mean_pos = position_pipeline.final_position.abs().mean()
    max_pos = position_pipeline.final_position.abs().max()
    current_pos = position_pipeline.final_position.iloc[-1]
    curr_price = instrument.price_data.data['Close'].iloc[-1]
    df[ticker] = [mean_pos, max_pos, current_pos, curr_price, abs(curr_price * current_pos),portfolio.instrument_weights.weights[ticker]]

df = pd.DataFrame(df).T
df.columns = ["Mean Pos", "Max Pos", "Current Pos", "Current Price", "Total Pos Value","Instrument Weight"]
df, df.sum()

(       Mean Pos  Max Pos  Current Pos  Current Price  Total Pos Value  \
 SPY   15.694679     39.0          2.0     676.469971      1352.939941   
 TLT   27.736941     52.0         22.0      88.220001      1940.840027   
 GLD   17.335411     48.0          5.0     398.570007      1992.850037   
 DBC  109.298998    211.0         88.0      22.690001      1996.720047   
 VNQ   47.681486    185.0         22.0      88.930000      1956.460007   
 
      Instrument Weight  
 SPY                0.2  
 TLT                0.2  
 GLD                0.2  
 DBC                0.2  
 VNQ                0.2  ,
 Mean Pos              217.747514
 Max Pos               535.000000
 Current Pos           139.000000
 Current Price        1274.879980
 Total Pos Value      9239.810059
 Instrument Weight       1.000000
 dtype: float64)